In [ ]:
from google.cloud import storage
from io import BytesIO
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

def upload_to_gcs(df: pd.DataFrame, bucket_name: str, blob_name: str):
    """
    Uploads a DataFrame to GCS as a Parquet file in the 'cleaned/' folder.
    """
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(f"cleaned/{blob_name}.parquet")

    parquet_buffer = BytesIO()
    df.to_parquet(parquet_buffer, index=False, engine='pyarrow')
    blob.upload_from_string(parquet_buffer.getvalue(), content_type='application/octet-stream')

    print(f"Uploaded {blob_name}.parquet to GCS bucket {bucket_name} in the 'cleaned/' folder.")

def extract_datetime_features(df: pd.DataFrame):
    """
    Extracts components from datetime columns and appends them as new columns.
    """
    new_dates_df = pd.DataFrame()
    for col in df.select_dtypes(include=['datetime64']).columns:
        x = df[col]
        if col in ['pickup_datetime', 'request_datetime']:
            new_dates_df[f'{col}_year'] = x.dt.year
            new_dates_df[f'{col}_month'] = x.dt.month
            new_dates_df[f'{col}_day_of_month'] = x.dt.day
            new_dates_df[f'{col}_week_day'] = x.dt.weekday  # 0=Monday, 6=Sunday
            new_dates_df[f'{col}_hour'] = x.dt.hour
            new_dates_df[f'{col}_minute'] = x.dt.minute
            new_dates_df[f'{col}_time_of_day'] = x.dt.strftime("%p").map({'AM': 1, 'PM': 0})

    df = df.join(new_dates_df)
    #Drop date time columns that are tranformed and columns that can be calculated
    df.drop(columns=['pickup_datetime', 'request_datetime', 'on_scene_datetime','dropoff_datetime'], inplace=True, errors='ignore')
    print("Extracted datetime features, updated DataFrame, and dropped pickup_datetime', 'request_datetime', 'on_scene_datetime','dropoff_datetime'.")
    return df

def perform_cleaning(df: pd.DataFrame):
    """
    Cleans the DataFrame by removing duplicates, handling outliers, and imputing missing values.
    """
    df.drop_duplicates(inplace=True)
    print("Removed duplicate rows.")

    df.drop(['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'shared_request_flag'],
            axis=1, inplace=True, errors='ignore')
    print("Dropped unnecessary columns.")

    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        mean = df[col].mean()
        std_dev = df[col].std()
        df[col] = df[col].apply(lambda x: np.nan if x < 0 or abs(x - mean) > 2 * std_dev else x)
    print("Replaced negative values and outliers beyond 2 standard deviations with NaN.")

    imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
    df[df.select_dtypes(include=['int64', 'float64']).columns] = imp_mean.fit_transform(df.select_dtypes(include=['int64', 'float64']))
    print("Imputed missing values with column mean.")

    return df

def main_cleaning():
    """
    Cleans and uploads trip data from GCS landing zone to cleanednew folder in the bucket.
    """
    bucket_name = 'my-bigdata-project-kc'
    storage_client = storage.Client()
    blobs = storage_client.list_blobs(bucket_name, prefix="landing/")

    parquet_blobs = [blob for blob in blobs if blob.name.endswith('.parquet')]

    for blob in parquet_blobs:
        print(f"Processing file {blob.name} ({blob.size} bytes)")
        original_df = pd.read_parquet(BytesIO(blob.download_as_bytes()))

        cleaned_df = perform_cleaning(original_df.copy())
        cleaned_df = extract_datetime_features(cleaned_df)

        upload_to_gcs(cleaned_df, bucket_name, blob.name.replace('landing/', '').replace('.parquet', ''))

        if blob.name == "landing/yellow_tripdata_2022-01.parquet":
            break

if __name__ == "__main__":
    main_cleaning()



